RREF (Gauss-Jordan Elimination)


In [1]:
import numpy as np

define row elementary row operations

In [2]:
def switch_row(A, i, j):
  A = A.copy()
  A[[i, j]] = A[[j, i]]
  return A

def scale_row(A, i, scalar):
  A = A.copy()
  if scalar == 0:
    raise ValueError("Cannot scale a row by zero")
  else:
    A[i] = scalar * A[i]
  return A

def add_scaled_row(A, target, source, scalar):
  A = A.copy()
  A[target] = A[target] + scalar * A[source]
  return A

check elementary row operation works well

In [5]:
A = np.array([[2, -1, 3, -1], [1, 0, 2, 1], [1, -1, 1, -2]], float)
B = switch_row(A, 1,2)
C = scale_row(A, 1,3)
D = add_scaled_row(A, 0, 1, 2)

1. Define variables

In [6]:
def GJ(matrix):

  # 불러온 matrix값은 matrix 여야 함 (i.e., 2D array여야 함)
  if matrix.ndim != 2:
    raise ValueError(f"Input must be a 2D array (matrix), but got {matrix.ndim}D.")

  pivot_i = 0 # 찾은 pivot 개수
  n_rows  = matrix.shape[0] # 주어진 matrix 의 row 수
  n_cols = matrix.shape[1] # 주어진 marix의 column 수
  rref_history = [matrix.copy()] # row operation 할 때마다 history로 저장
  n_row_operations = 0  # RREF 만들때까지 몇 번의 row operation 했는지 count

2.  column 에서 pivot 찾기, nonzero element 가 없으면, 다음 column으로, 찾았으면 pivot_i 번째 row 와 row switch

In [7]:
def GJ(matrix):

  # 불러온 matrix값은 matrix 여야 함 (i.e., 2D array여야 함)
  if matrix.ndim != 2:
    raise ValueError(f"Input must be a 2D array (matrix), but got {matrix.ndim}D.")

  pivot_i = 0 # 찾은 pivot 개수
  n_rows  = matrix.shape[0] # 주어진 matrix 의 row 수
  n_cols = matrix.shape[1] # 주어진 marix의 column 수
  rref_history = [matrix.copy()] # row operation 할 때마다 history로 저장
  n_row_operations = 0  # RREF 만들때까지 몇 번의 row operation 했는지 count

  ################################################
  for cols in range(n_cols):
    if pivot_i == n_rows: # column 이 row 보다 많을 때, 가능한 pivot 수에 따라서 다음 column은 조사할 필요 없음.
      break

    # column 안에 nonzero element 있는지 확인 (np.nonzero)
    non_zero_order = np.nonzero(matrix[pivot_i:, cols])[0]

    if len(non_zero_order) == 0: # nonzero element가 없으면 다음 column으로
      continue

    elif len(non_zero_order) > 0: # nonzero element가 있으면 row 위치 switch, (*스위치를 꼭 해야하는 것은 아님)
      matrix = switch_row(matrix, pivot_i, pivot_i + non_zero_order[0])

      ### row operation 하였으면 다음과 같이 history와 row operation count 추가
      n_row_operations = n_row_operations + 1
      rref_history.append(matrix.copy())

3. 찾은 pivot 이 1이 되도록 scaling

In [12]:
matrix = scale_row(matrix, pivot_i, 1 / matrix[pivot_i, cols])

NameError: name 'matrix' is not defined

4. pivot의 위 아래가 0이 되도록

In [13]:
    for rows in range(n_rows):
      if rows == pivot_i: # pivot 순서의 row 에 대해서는 계산할 필요없음.
        continue
      elif matrix[rows, cols] == 0: # 이미 element 가 0 이면 해당 row 에 대해서는 row operation 할 필요 없음
        continue
      else:
        matrix = add_scaled_row(matrix, rows, pivot_i, -matrix[rows, cols])  # pivot이 위치한 column의 element가 0이 되도록 계산
        n_row_operations = n_row_operations + 1

    rref_history.append(matrix.copy()) # 여기에서는 RREF history에 마지막 결과만 저장

NameError: name 'n_rows' is not defined

5. pivot 을 찾았고 계산을 다하였으면, pivot_i +1, 최종 형태

In [19]:
def GJ(matrix):

  matrix = matrix.astype(float).copy()

  if matrix.ndim != 2:
    raise ValueError(f"Input must be a 2D array (matrix), but got {matrix.ndim}D.")

  pivot_i = 0
  n_rows = matrix.shape[0]
  n_cols = matrix.shape[1]

  rref_history = [matrix.copy()]
  operation_history = ["start"]
  n_row_operations = 0

  for cols in range(n_cols):
    if pivot_i == n_rows:
      break

    non_zero_order = np.nonzero(matrix[pivot_i:, cols])[0]

    if len(non_zero_order) == 0:
      continue

    pivot_row = pivot_i + non_zero_order[0]

    # 1. switch 필요할 때만
    if pivot_row != pivot_i:
      matrix = switch_row(matrix, pivot_i, pivot_row)
      n_row_operations += 1
      rref_history.append(matrix.copy())
      operation_history.append(f"R{pivot_i} <-> R{pivot_row}")

    # 2. pivot을 1로 만들기
    pivot_value = matrix[pivot_i, cols]
    if pivot_value != 1:
      matrix = scale_row(matrix, pivot_i, 1 / pivot_value)
      n_row_operations += 1
      rref_history.append(matrix.copy())
      operation_history.append(f"R{pivot_i} <- (1/{pivot_value})R{pivot_i}")

    # 3. 같은 column의 다른 row들을 하나씩 0으로 만들기
    for rows in range(n_rows):
      if rows == pivot_i:
        continue
      elif matrix[rows, cols] == 0:
        continue
      else:
        factor = -matrix[rows, cols]
        matrix = add_scaled_row(matrix, rows, pivot_i, factor)
        n_row_operations += 1
        rref_history.append(matrix.copy())
        operation_history.append(f"R{rows} <- R{rows} + ({factor})R{pivot_i}")

    pivot_i += 1

  return matrix, rref_history, operation_history, n_row_operations

결과 체크

In [20]:
A = np.array([
    [0, 2, -2, 2, 3],
    [1, 2, -1, 3, 5],
    [1, 2, -3, 1, 1],
    [3, 6, 8, 8, 7]
], float)

RREF, history, ops, count = GJ(A)

print("RREF")
print(RREF)
print()
print("Total elementary operations:", count)
print()

for i, (op, mat) in enumerate(zip(ops, history)):
    print(f"order {i} : {op}")
    print(mat)
    print()

RREF
[[ 1.   0.   0.   0.   0. ]
 [ 0.   1.   0.   0.  -1.5]
 [ 0.   0.   1.   0.  -0.5]
 [-0.  -0.  -0.   1.   2.5]]

Total elementary operations: 12

order 0 : start
[[ 0.  2. -2.  2.  3.]
 [ 1.  2. -1.  3.  5.]
 [ 1.  2. -3.  1.  1.]
 [ 3.  6.  8.  8.  7.]]

order 1 : R0 <-> R1
[[ 1.  2. -1.  3.  5.]
 [ 0.  2. -2.  2.  3.]
 [ 1.  2. -3.  1.  1.]
 [ 3.  6.  8.  8.  7.]]

order 2 : R2 <- R2 + (-1.0)R0
[[ 1.  2. -1.  3.  5.]
 [ 0.  2. -2.  2.  3.]
 [ 0.  0. -2. -2. -4.]
 [ 3.  6.  8.  8.  7.]]

order 3 : R3 <- R3 + (-3.0)R0
[[ 1.  2. -1.  3.  5.]
 [ 0.  2. -2.  2.  3.]
 [ 0.  0. -2. -2. -4.]
 [ 0.  0. 11. -1. -8.]]

order 4 : R1 <- (1/2.0)R1
[[ 1.   2.  -1.   3.   5. ]
 [ 0.   1.  -1.   1.   1.5]
 [ 0.   0.  -2.  -2.  -4. ]
 [ 0.   0.  11.  -1.  -8. ]]

order 5 : R0 <- R0 + (-2.0)R1
[[ 1.   0.   1.   1.   2. ]
 [ 0.   1.  -1.   1.   1.5]
 [ 0.   0.  -2.  -2.  -4. ]
 [ 0.   0.  11.  -1.  -8. ]]

order 6 : R2 <- (1/-2.0)R2
[[ 1.   0.   1.   1.   2. ]
 [ 0.   1.  -1.   1.   1.5]
 [-0.  -0

RREF 만드는 과정에서 사용한 elementary operation 을 통해서 역행렬 구하는 알고리즘으로 확장 가능.(과제)

In [21]:
def inverse_by_GJ(A):
+피벗 엔트리가 1이면 스킵?하는 코드 작성해야함
  A = A.astype(float).copy()

  if A.ndim != 2:
    raise ValueError("Input must be a 2D matrix.")

  n_rows = A.shape[0]
  n_cols = A.shape[1]

  if n_rows != n_cols:
    raise ValueError("Inverse only exists for square matrices.")

  # [A | I] 만들기
  I = np.eye(n_rows)
  aug = np.hstack([A, I])

  pivot_i = 0
  rref_history = [aug.copy()]
  operation_history = ["start"]
  n_row_operations = 0

  # 왼쪽 A 부분만 보면서 pivot 찾기
  for cols in range(n_cols):

    if pivot_i == n_rows:
      break

    non_zero_order = np.nonzero(aug[pivot_i:, cols])[0]

    if len(non_zero_order) == 0:
      continue

    pivot_row = pivot_i + non_zero_order[0]

    # 1. switch
    if pivot_row != pivot_i:
      aug = switch_row(aug, pivot_i, pivot_row)
      n_row_operations += 1
      rref_history.append(aug.copy())
      operation_history.append(f"R{pivot_i} <-> R{pivot_row}")

    # 2. scale
    pivot_value = aug[pivot_i, cols]

    if pivot_value != 1:
      aug = scale_row(aug, pivot_i, 1 / pivot_value)
      n_row_operations += 1
      rref_history.append(aug.copy())
      operation_history.append(f"R{pivot_i} <- (1/{pivot_value})R{pivot_i}")

    # 3. add
    for rows in range(n_rows):
      if rows == pivot_i:
        continue
      elif aug[rows, cols] == 0:
        continue
      else:
        factor = -aug[rows, cols]
        aug = add_scaled_row(aug, rows, pivot_i, factor)
        n_row_operations += 1
        rref_history.append(aug.copy())
        operation_history.append(f"R{rows} <- R{rows} + ({factor})R{pivot_i}")

    pivot_i += 1

  left = aug[:, :n_cols]
  right = aug[:, n_cols:]

  # 왼쪽이 I가 아니면 inverse 없음
  if not np.allclose(left, np.eye(n_rows)):
    raise ValueError("This matrix is singular, so inverse does not exist.")

  A_inv = right

  return A_inv, aug, rref_history, operation_history, n_row_operations

In [24]:
A = np.array([
    [1, 2, 1],
    [2, 5, 3],
    [1, 0, 8]
], float)

A_inv, aug_final, history, ops, count = inverse_by_GJ(A)

print("A inverse")
print(A_inv)
print()

print("Final augmented matrix [I | A_inv]")
print(aug_final)
print()

print("Total elementary operations:", count)
print()

for i, (op, mat) in enumerate(zip(ops, history)):
    print(f"order {i} : {op}")
    print(mat)
    print()

print("Check A @ A_inv")
print(A @ A_inv)

A inverse
[[ 4.44444444 -1.77777778  0.11111111]
 [-1.44444444  0.77777778 -0.11111111]
 [-0.55555556  0.22222222  0.11111111]]

Final augmented matrix [I | A_inv]
[[ 1.          0.          0.          4.44444444 -1.77777778  0.11111111]
 [ 0.          1.          0.         -1.44444444  0.77777778 -0.11111111]
 [ 0.          0.          1.         -0.55555556  0.22222222  0.11111111]]

Total elementary operations: 7

order 0 : start
[[1. 2. 1. 1. 0. 0.]
 [2. 5. 3. 0. 1. 0.]
 [1. 0. 8. 0. 0. 1.]]

order 1 : R1 <- R1 + (-2.0)R0
[[ 1.  2.  1.  1.  0.  0.]
 [ 0.  1.  1. -2.  1.  0.]
 [ 1.  0.  8.  0.  0.  1.]]

order 2 : R2 <- R2 + (-1.0)R0
[[ 1.  2.  1.  1.  0.  0.]
 [ 0.  1.  1. -2.  1.  0.]
 [ 0. -2.  7. -1.  0.  1.]]

order 3 : R0 <- R0 + (-2.0)R1
[[ 1.  0. -1.  5. -2.  0.]
 [ 0.  1.  1. -2.  1.  0.]
 [ 0. -2.  7. -1.  0.  1.]]

order 4 : R2 <- R2 + (2.0)R1
[[ 1.  0. -1.  5. -2.  0.]
 [ 0.  1.  1. -2.  1.  0.]
 [ 0.  0.  9. -5.  2.  1.]]

order 5 : R2 <- (1/9.0)R2
[[ 1.          0.  